<a href="https://colab.research.google.com/github/bounouarafarah/OIBSIP/blob/main/DataScience-Task4-UnemploymentAnalysis/unemployment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
import zipfile

uploaded = files.upload()
fname = list(uploaded.keys())[0]

# Si c'est un fichier ZIP avec plusieurs fichiers, on extrait celui de notre choix
if fname.endswith('.zip'):
    with zipfile.ZipFile(fname, 'r') as zip_ref:
        # On extrait 'Unemployment in India.csv'
        zip_ref.extract('Unemployment in India.csv', '/content/')
        csv_fname = 'Unemployment in India.csv'
else:
    csv_fname = fname

df = pd.read_csv(csv_fname)

df.columns = df.columns.str.strip()
df = df.rename(columns={
    "Estimated Unemployment Rate (%)": "UnemploymentRate",
    "Estimated Employed": "Employed",
    "Estimated Labour Participation Rate (%)": "LabourParticipation"})
df["Region"] = df["Region"].str.strip()
df["Date"] = pd.to_datetime(df["Date"].str.strip(), dayfirst=True)

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head()

In [ ]:
print(df.groupby("Region")["UnemploymentRate"].mean().sort_values(ascending=False).round(2))

monthly = df.groupby(df["Date"].dt.to_period("M"))["UnemploymentRate"].mean()
monthly.plot(figsize=(10, 4), marker="o", title="Average unemployment rate by month")
plt.ylabel("Unemployment rate (%)"); plt.show()

## Observations: regional and monthly trends

- The raw data had 768 rows, but 28 of them were completely empty; I removed them before the analysis.
- Averaged over the period, the highest unemployment rates are in Tripura (28.4%), Haryana (26.3%), Jharkhand (20.6%), Bihar (18.9%) and Himachal Pradesh (18.5%), while Meghalaya (4.8%) and Odisha (5.7%) have the lowest.
- The monthly curve rises sharply to about 24% in spring 2020, which coincides with the national COVID-19 lockdown.

## Observations: unemployment over time

- Before the lockdown, unemployment in the four selected states was fairly stable, between about 5% (Maharashtra, Tamil Nadu) and 18% (Delhi).
- After the lockdown of 25 March 2020 (red dashed line), the rates jump: Bihar and Tamil Nadu reach about 50%, Delhi about 34% and Maharashtra about 20%.
- The shock is very uneven across states. Delhi peaks later and lower than Bihar and Tamil Nadu, and all four states drop again in June 2020 as activity restarts.

In [ ]:
before = df.shape[0]
df = df.dropna().reset_index(drop=True)
print("Rows before:", before, "| after:", df.shape[0])
print(df.isnull().sum().sum(), "null values left")
print(df["Date"].min(), "->", df["Date"].max())

In [ ]:
states = ["Delhi", "Maharashtra", "Tamil Nadu", "Bihar"]
plt.figure(figsize=(11, 5))
for s in states:
    d = df[df["Region"] == s].groupby("Date")["UnemploymentRate"].mean()
    plt.plot(d.index, d.values, marker="o", label=s)
plt.axvline(pd.Timestamp("2020-03-25"), color="red", linestyle="--", label="Lockdown (25 Mar 2020)")
plt.title("Unemployment rate over time"); plt.ylabel("%"); plt.legend(); plt.show()

In [ ]:
top10 = df.groupby("Region")["UnemploymentRate"].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=top10.values, y=top10.index, color="steelblue")
plt.xlabel("Average unemployment rate (%)"); plt.title("Top 10 states"); plt.show()

## Observations: top 10 states

- The highest average unemployment rates are in Tripura (28.4%), Haryana (26.3%), Jharkhand (20.6%), Bihar (18.9%) and Himachal Pradesh (18.5%), followed by Delhi, Jammu & Kashmir, Chandigarh, Rajasthan and Uttar Pradesh.
- These averages include the COVID months, so states hit hard by the lockdown are pushed upward. The ranking mixes structural unemployment and the pandemic shock.

In [ ]:
sns.heatmap(df[["UnemploymentRate", "Employed", "LabourParticipation"]].corr(), annot=True, cmap="Blues")
plt.show()

## Observations: correlation heatmap

- The unemployment rate is weakly and negatively correlated with the number of employed people (-0.22): when unemployment rises, employment tends to fall, but the link is weak.
- The correlations with the labour participation rate are close to zero (0.0026 with unemployment, 0.011 with employment).
- These weak values come from the fact that the data mixes states of very different sizes. A state-by-state analysis would probably show stronger relationships.

In [ ]:
split = pd.Timestamp("2020-03-25")
df["Period"] = np.where(df["Date"] < split, "Pre-COVID", "Post-COVID")
cmp = df.groupby("Period")[["UnemploymentRate", "Employed", "LabourParticipation"]].mean().round(2)
print(cmp)
cmp["UnemploymentRate"].plot(kind="bar", color=["#d9534f", "#5b8db8"])
plt.ylabel("Average unemployment rate (%)"); plt.xticks(rotation=0); plt.show()

## Observations: pre-COVID vs post-COVID

| Period | Unemployment rate (%) | Employed (mean) | Labour participation (%) |
|---|---|---|---|
| Pre-COVID | 9.51 | 7,466,028 | 43.89 |
| Post-COVID | 17.77 | 6,517,203 | 39.33 |

- The average unemployment rate nearly doubled after the lockdown (+8.3 points), the number of employed people fell by about 12.7% and labour participation dropped by 4.6 points.
- Note: the post-COVID period only covers March to June 2020 (4 months) against 11 months before, so this is a short-term shock and not a long-term trend.
- **Conclusion:** the COVID-19 lockdown caused a sharp but uneven rise in unemployment in India, with the strongest impact in Bihar and Tamil Nadu.